# 01 — Embeddings & Semantic Search

This notebook explores the foundational building block of any RAG pipeline: **text embeddings**.

We'll learn:
- What embeddings look like (vectors of numbers)
- How cosine similarity measures semantic closeness
- How to build a mini semantic search engine in ~10 lines of code

**Model used:** `all-MiniLM-L6-v2` — a fast, open-source embedding model (384 dimensions).

In [3]:
!pip install sentence_transformers

  Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl.metadata (60 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.4 MB/s  0:00:0058.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 45.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 65.0 MB/s  0:00:00
Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl (5.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 33.4 MB/s  0:00:02a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 27.3 MB/s  0:00:000.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 15.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 15.4 MB/s  0:00:002.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 22.0 MB/s  0:00:0021.8 MB/s eta 0:00:01


In [4]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Load embedding model (~80MB download on first run)
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

/Users/nischal/Desktop/Vault/03_Projects/RAG/project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Embedding dimension: 384


## What does an embedding look like?

An embedding is just a list of numbers — a **vector** — that captures the *meaning* of a piece of text.

In [6]:
text = "The cat sat on the mat"
embedding = model.encode(text)

print(f"Type: {type(embedding)}")
print(f"Shape: {embedding.shape}")
print(f"First 10 values: {embedding[:10].round(4)}")
print(f"Min: {embedding.min():.4f}, Max: {embedding.max():.4f}")

Type: <class 'numpy.ndarray'>
Shape: (384,)
First 10 values: [ 0.1304 -0.0119 -0.0281  0.0512 -0.056   0.0302  0.0302  0.0247 -0.0184
  0.0588]
Min: -0.1684, Max: 0.1426


## Cosine similarity: measuring meaning

Similar meaning → nearby vectors → high cosine similarity (close to 1.0).

In [7]:
pairs = [
    ("How do I reset my password?", "I forgot my login credentials"),
    ("How do I reset my password?", "The weather is nice today"),
    ("The dog ran quickly", "A canine sprinted fast"),
    ("Quantum physics research", "Company vacation policy"),
]

print(f"{'Sentence A':<35} {'Sentence B':<35} {'Similarity':>10}")
print("-" * 82)

for a, b in pairs:
    emb_a = model.encode(a)
    emb_b = model.encode(b)
    sim = util.cos_sim(emb_a, emb_b).item()
    print(f"{a:<35} {b:<35} {sim:>10.4f}")

Sentence A                          Sentence B                          Similarity
----------------------------------------------------------------------------------
How do I reset my password?         I forgot my login credentials           0.6776
How do I reset my password?         The weather is nice today              -0.0425
The dog ran quickly                 A canine sprinted fast                  0.8664
Quantum physics research            Company vacation policy                -0.0208


## Mini semantic search engine

This is the **retrieval core of RAG** — in ~10 lines of code.

In [11]:
# Our "knowledge base" — imagine these are chunks from company documents
documents = [
    "The company offers 12 weeks of paid parental leave.",
    "Employees can expense up to $500 per year for learning and development.",
    "Remote work is allowed up to 3 days per week.",
    "Health insurance covers dental and vision.",
    "The office is closed on all federal holidays.",
    "Annual performance reviews happen in Q4.",
    "The 401(k) plan matches up to 4% of salary.",
    "Unlimited PTO is available after the first year of employment.",
]

# Embed all documents (done once, ahead of time in a real system)
doc_embeddings = model.encode(documents)

In [14]:
def search(query: str, top_k: int = 3):
    """Search documents by semantic similarity."""
    query_embedding = model.encode(query)
    similarities = util.cos_sim(query_embedding, doc_embeddings)[0]

    top_indices = similarities.argsort(descending=True)[:top_k]

    print(f"Query: '{query}'\n")
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank}. [{similarities[idx]:.3f}] {documents[idx]}")
    print()


# Try different queries
search("Can I work from home?")
search("What happens when I have a baby?")
search("How much vacation do I get?")
search("Does the company help with retirement savings?")
search("What is the salary range for my position?")
search("Is coffee provided at work?")

Query: 'Can I work from home?'

  1. [0.417] Remote work is allowed up to 3 days per week.
  2. [0.207] Unlimited PTO is available after the first year of employment.
  3. [0.201] The company offers 12 weeks of paid parental leave.

Query: 'What happens when I have a baby?'

  1. [0.194] The company offers 12 weeks of paid parental leave.
  2. [0.043] Health insurance covers dental and vision.
  3. [0.010] The 401(k) plan matches up to 4% of salary.

Query: 'How much vacation do I get?'

  1. [0.406] The company offers 12 weeks of paid parental leave.
  2. [0.342] Employees can expense up to $500 per year for learning and development.
  3. [0.269] Remote work is allowed up to 3 days per week.

Query: 'Does the company help with retirement savings?'

  1. [0.417] The 401(k) plan matches up to 4% of salary.
  2. [0.269] Unlimited PTO is available after the first year of employment.
  3. [0.268] The company offers 12 weeks of paid parental leave.

Query: 'What is the salary range for my p

## Using our package module

The same logic, but using the `rag_pipeline.embeddings` module we'll build up over the course:

In [13]:
import sys
sys.path.insert(0, "../src")

from rag_pipeline.embeddings import load_model, rank_by_similarity

model = load_model()
results = rank_by_similarity("Can I work from home?", documents, model, top_k=3)

for doc, score in results:
    print(f"[{score:.3f}] {doc}")

[0.417] Remote work is allowed up to 3 days per week.
[0.207] Unlimited PTO is available after the first year of employment.
[0.201] The company offers 12 weeks of paid parental leave.


## Key takeaways

1. **Embeddings** convert text meaning into vectors — numbers you can do math on
2. **Cosine similarity** tells you how close two meanings are (0 = unrelated, 1 = identical)
3. **Semantic search** beats keyword search because it understands meaning, not just words
4. This is the **retrieval engine of RAG** — the "R" in the name

**Next notebook:** Document chunking — how to split real PDFs into pieces that embed well.